# SDOCX Format — Container Structure

Samsung Notes stores handwritten notes as `.sdocx` files. Despite the name suggesting
a document format, these are actually **ZIP archives** containing binary stroke data,
metadata, and media. The format is produced by the **Samsung S-Pen SDK**, used across
Galaxy Note and Galaxy Tab devices.

This notebook documents the container format by examining a real handwritten note.

In [1]:
from pathlib import Path

SAMPLE = Path("../samples/handwritten.sdocx")

In [2]:
import zipfile
import struct
import hashlib
import datetime

In [3]:
def hexdump(data: bytes, offset: int = 0, limit: int = 0) -> str:
    """Format binary data as an annotated hex dump."""
    lines = []
    n = len(data) if limit == 0 else min(len(data), limit)
    for i in range(0, n, 16):
        chunk = data[i : i + 16]
        hex_part = " ".join(f"{b:02x}" for b in chunk)
        ascii_part = "".join(chr(b) if 32 <= b < 127 else "\u00b7" for b in chunk)
        lines.append(f"  {offset + i:04x}  {hex_part:<48s} {ascii_part}")
    if limit and len(data) > limit:
        lines.append(f"  ... ({len(data) - limit} more bytes)")
    return "\n".join(lines)

## ZIP Contents

The `.sdocx` file is a standard ZIP archive. Let's list everything inside.

In [4]:
PURPOSES = {
    "pageIdInfo.dat": "Page UUID registry",
    ".page": "Stroke data (geometry + attributes)",
    "mediaInfo.dat": "Media file manifest",
    ".spi": "Page thumbnail (Samsung proprietary)",
    "note.note": "Note metadata & settings",
    "end_tag.bin": "Document footer & timestamps",
}

with zipfile.ZipFile(SAMPLE) as z:
    print(f"{'File':<45} {'Size':>8}  {'Ratio':>6}  Purpose")
    print("\u2500" * 90)
    for info in z.infolist():
        ratio = f"{(1 - info.compress_size / info.file_size) * 100:.0f}%" if info.file_size else "\u2014"
        purpose = next((v for k, v in PURPOSES.items() if k in info.filename), "?")
        print(f"{info.filename:<45} {info.file_size:>8,}  {ratio:>6}  {purpose}")

File                                              Size   Ratio  Purpose
──────────────────────────────────────────────────────────────────────────────────────────
pageIdInfo.dat                                     140     11%  Page UUID registry
a87cbcbc-43ce-11ee-9848-0b8ce2c9daf6.page     4,487,877     68%  Stroke data (geometry + attributes)
media/mediaInfo.dat                                131     14%  Media file manifest
media/0@page_0000000.spi                       453,791      0%  Page thumbnail (Samsung proprietary)
note.note                                        1,741     64%  Note metadata & settings
end_tag.bin                                        144     37%  Document footer & timestamps


In [5]:
with zipfile.ZipFile(SAMPLE) as z:
    files = {info.filename: z.read(info.filename) for info in z.infolist()}

### File tree

```
handwritten.sdocx (ZIP)
├── pageIdInfo.dat          Page UUID with integrity hashes (140 B)
├── <uuid>.page             Stroke coordinates + attributes (~4.5 MB)
├── media/
│   ├── mediaInfo.dat       Media manifest with SHA-256 hash (131 B)
│   └── 0@page_0000000.spi  Page thumbnail, proprietary format (~454 KB)
├── note.note               Title, pen tools, background color (1.7 KB)
└── end_tag.bin             Timestamps, SDK marker (144 B)
```

**Conventions across all files:**
- All integers are **little-endian**
- Strings use **UTF-16LE** (in binary fields) or plain **ASCII** (UUIDs, hashes)
- Timestamps are **int64 milliseconds since Unix epoch** (UTC)
- Footer markers identify the SDK: `"Document for S-Pen SDK"`, `"Page for SAMSUNG S-Pen SDK"`, `"EOFX"`

---

## end_tag.bin

The document footer (144 bytes). Contains **creation and modification timestamps** as
int64 milliseconds since Unix epoch, and ends with the ASCII marker `"Document for S-Pen SDK"`.

| Offset | Size | Type | Content |
|--------|------|------|---------|
| `0x48` | 8 | i64 | Created timestamp (ms since epoch) |
| `0x50` | 8 | i64 | Modified timestamp (ms since epoch) |
| `0x7A` | 22 | ASCII | `"Document for S-Pen SDK"` |

In [6]:
data = files["end_tag.bin"]
print(f"end_tag.bin \u2014 {len(data)} bytes\n")

ts_created = struct.unpack_from("<q", data, 0x48)[0]
ts_modified = struct.unpack_from("<q", data, 0x50)[0]
created = datetime.datetime.fromtimestamp(ts_created / 1000, tz=datetime.timezone.utc)
modified = datetime.datetime.fromtimestamp(ts_modified / 1000, tz=datetime.timezone.utc)

print(f"Created:   {created}")
print(f"Modified:  {modified}")
print(f"Marker:    {data[-22:].decode('ascii')!r}")

end_tag.bin — 144 bytes

Created:   2023-08-26 05:09:52.821000+00:00
Modified:  2023-08-30 06:06:10.787000+00:00
Marker:    'Document for S-Pen SDK'


In [7]:
print(hexdump(data))

  0000  8e 00 a0 0f 00 00 00 00 95 c4 2e ab 87 36 06 00  ··········.··6··
  0010  00 00 00 00 00 00 38 07 00 00 00 f0 f4 45 00 00  ······8······E··
  0020  ff ff ff ff ff ff ff ff 00 00 a0 0f 00 00 c3 35  ···············5
  0030  a9 78 cc 03 06 00 00 00 00 00 01 00 00 00 00 00  ·x··············
  0040  00 00 00 00 00 00 00 00 35 a8 3f 30 8a 01 00 00  ········5·?0····
  0050  63 a3 0c 45 8a 01 00 00 00 00 00 00 00 00 00 00  c··E············
  0060  00 00 02 00 00 00 02 00 00 00 ff ff ff ff ff ff  ················
  0070  ff ff 00 00 00 00 00 00 00 00 44 6f 63 75 6d 65  ··········Docume
  0080  6e 74 20 66 6f 72 20 53 2d 50 65 6e 20 53 44 4b  nt for S-Pen SDK


---

## pageIdInfo.dat

The page registry (140 bytes). Contains a **page UUID** as a length-prefixed UTF-16LE string,
sandwiched between two 32-byte values (likely integrity hashes or checksums).

| Offset | Size | Type | Content |
|--------|------|------|---------|
| `0x00` | 32 | bytes | Hash/checksum A |
| `0x20` | 2 | u16 | Page count |
| `0x22` | 2 | u16 | UUID string length (chars) |
| `0x24` | 72 | UTF-16LE | Page UUID (36 chars × 2 bytes) |
| `0x6C` | 32 | bytes | Hash/checksum B |

In [8]:
data = files["pageIdInfo.dat"]
print(f"pageIdInfo.dat \u2014 {len(data)} bytes\n")

hash_a = data[0x00:0x20]
count = struct.unpack_from("<H", data, 0x20)[0]
str_len = struct.unpack_from("<H", data, 0x22)[0]
uuid = data[0x24 : 0x24 + str_len * 2].decode("utf-16-le")
hash_b = data[0x6C:0x8C]

page_filename = [k for k in files if k.endswith(".page")][0]

print(f"Page count:    {count}")
print(f"Page UUID:     {uuid}")
print(f".page file:    {page_filename}")
print(f"UUID matches:  {uuid == page_filename.removesuffix('.page')}")
print(f"Hash A:        {hash_a.hex()}")
print(f"Hash B:        {hash_b.hex()}")

pageIdInfo.dat — 140 bytes

Page count:    1
Page UUID:     a87cbcbc-43ce-11ee-9848-0b8ce2c9daf6
.page file:    a87cbcbc-43ce-11ee-9848-0b8ce2c9daf6.page
UUID matches:  True
Hash A:        bcd9cd528172b68b6e4179822dcb99b3f5313aaf0bcc1d1c045bcbfe10a50413
Hash B:        a75114fea7fa3f1e566e25a621a8c736730f6b3798c95a381b400888c5e86ca8


In [9]:
print(hexdump(data))

  0000  bc d9 cd 52 81 72 b6 8b 6e 41 79 82 2d cb 99 b3  ···R·r··nAy·-···
  0010  f5 31 3a af 0b cc 1d 1c 04 5b cb fe 10 a5 04 13  ·1:······[······
  0020  01 00 24 00 61 00 38 00 37 00 63 00 62 00 63 00  ··$·a·8·7·c·b·c·
  0030  62 00 63 00 2d 00 34 00 33 00 63 00 65 00 2d 00  b·c·-·4·3·c·e·-·
  0040  31 00 31 00 65 00 65 00 2d 00 39 00 38 00 34 00  1·1·e·e·-·9·8·4·
  0050  38 00 2d 00 30 00 62 00 38 00 63 00 65 00 32 00  8·-·0·b·8·c·e·2·
  0060  63 00 39 00 64 00 61 00 66 00 36 00 a7 51 14 fe  c·9·d·a·f·6··Q··
  0070  a7 fa 3f 1e 56 6e 25 a6 21 a8 c7 36 73 0f 6b 37  ··?·Vn%·!··6s·k7
  0080  98 c9 5a 38 1b 40 08 88 c5 e8 6c a8              ··Z8·@····l·


---

## media/mediaInfo.dat

Media manifest (131 bytes). References the `.spi` thumbnail with its **UTF-16LE filename**
and an **ASCII hex SHA-256 hash** for integrity verification. Ends with the `EOFX` marker.

| Offset | Size | Type | Content |
|--------|------|------|---------|
| `0x0E` | 2 | u16 | Filename string length (chars) |
| `0x10` | var | UTF-16LE | Media filename |
| after filename | 64 | ASCII | SHA-256 hash (hex-encoded) |
| last 4 | 4 | ASCII | `"EOFX"` marker |

In [10]:
data = files["media/mediaInfo.dat"]
print(f"media/mediaInfo.dat \u2014 {len(data)} bytes\n")

str_len = struct.unpack_from("<H", data, 0x0E)[0]
filename = data[0x10 : 0x10 + str_len * 2].decode("utf-16-le")
hash_offset = 0x10 + str_len * 2
sha256_stored = data[hash_offset : hash_offset + 64].decode("ascii")

spi_key = f"media/{filename}"
sha256_actual = hashlib.sha256(files[spi_key]).hexdigest()

print(f"Filename:   media/{filename}")
print(f"SHA-256:    {sha256_stored}")
print(f"Verified:   {sha256_stored == sha256_actual}")
print(f"End marker: {data[-4:].decode('ascii')!r}")

media/mediaInfo.dat — 131 bytes

Filename:   media/0@page_0000000.spi
SHA-256:    64327fdf9a1940d1fdd7bfa7ec48c1ddbb2bb262d2653ff7de6f16b1555f660c
Verified:   True
End marker: 'EOFX'


In [11]:
print(hexdump(data))

  0000  52 14 00 00 01 00 75 00 00 00 00 00 00 00 12 00  R·····u·········
  0010  30 00 40 00 70 00 61 00 67 00 65 00 5f 00 30 00  0·@·p·a·g·e·_·0·
  0020  30 00 30 00 30 00 30 00 30 00 30 00 2e 00 73 00  0·0·0·0·0·0·.·s·
  0030  70 00 69 00 36 34 33 32 37 66 64 66 39 61 31 39  p·i·64327fdf9a19
  0040  34 30 64 31 66 64 64 37 62 66 61 37 65 63 34 38  40d1fdd7bfa7ec48
  0050  63 31 64 64 62 62 32 62 62 32 36 32 64 32 36 35  c1ddbb2bb262d265
  0060  33 66 66 37 64 65 36 66 31 36 62 31 35 35 35 66  3ff7de6f16b1555f
  0070  36 36 30 63 01 00 6e 33 2c ab 87 36 06 00 01 45  660c··n3,··6···E
  0080  4f 46 58                                         OFX


---

## note.note

Note-level metadata (1.7 KB). The most complex metadata file — it contains the **note title**
(UTF-16LE), **pen tool** class names (Java package paths like
`com.samsung.android.sdk.pen.pen.preload.FountainPen`), **layer UUIDs**, **page dimensions**,
and the **background color**.

The file uses a mix of fixed-header fields and variable-length TLV-style records.

In [12]:
data = files["note.note"]
print(f"note.note \u2014 {len(data)} bytes\n")

page_w = struct.unpack_from("<I", data, 0x28)[0]
page_h = struct.unpack_from("<I", data, 0x2C)[0]
print(f"Page dimensions: {page_w} \u00d7 {page_h}")

note.note — 1741 bytes

Page dimensions: 1848 × 7838


In [ ]:
# Scan for ASCII strings >= 4 chars
print("--- ASCII strings ---")
s, start = "", 0
for i, b in enumerate(data):
    if 32 <= b < 127:
        if not s:
            start = i
        s += chr(b)
    else:
        if len(s) >= 4:
            print(f"  0x{start:04x}: {s!r}")
        s = ""
if len(s) >= 4:
    print(f"  0x{start:04x}: {s!r}")

# Scan for UTF-16LE strings >= 3 chars
print("\n--- UTF-16LE strings ---")
i = 0
while i < len(data) - 3:
    if 32 <= data[i] < 127 and data[i + 1] == 0:
        start = i
        chars = []
        while i < len(data) - 1 and 32 <= data[i] < 127 and data[i + 1] == 0:
            chars.append(chr(data[i]))
            i += 2
        text = "".join(chars)
        if len(text) >= 3:
            print(f"  0x{start:04x}: {text!r}")
    else:
        i += 1

In [14]:
# Background color: pattern [18 00] [00 00 01 00 00 00] [R] [G] [B] [FF]
for i in range(len(data) - 12):
    if (
        data[i] == 0x18
        and data[i + 1] == 0x00
        and data[i + 2 : i + 8] == b"\x00\x00\x01\x00\x00\x00"
        and data[i + 11] == 0xFF
    ):
        r, g, b = data[i + 8], data[i + 9], data[i + 10]
        print(f"Background color: #{r:02x}{g:02x}{b:02x}  (R={r}, G={g}, B={b})")
        print(f"Found at offset:  0x{i:04x}")
        break

Background color: #252525  (R=37, G=37, B=37)
Found at offset:  0x02f1


---

## SPI Thumbnail

The `.spi` file is stored uncompressed (`STORED` method in the ZIP), suggesting it's
already in a compressed format. Let's check if it matches any standard image format.

In [15]:
spi_key = [k for k in files if k.endswith(".spi")][0]
spi = files[spi_key]
print(f"{spi_key} \u2014 {len(spi):,} bytes\n")

for fmt, check in {
    "PNG": spi[:4] == b"\x89PNG",
    "JPEG": spi[:2] == b"\xff\xd8",
    "BMP": spi[:2] == b"BM",
    "GIF": spi[:3] == b"GIF",
    "WEBP": spi[:4] == b"RIFF" and spi[8:12] == b"WEBP",
    "TIFF": spi[:2] in (b"II", b"MM"),
}.items():
    print(f"  {fmt}: {'MATCH' if check else '\u2014'}")

media/0@page_0000000.spi — 453,791 bytes

  PNG: —
  JPEG: —
  BMP: —
  GIF: —
  WEBP: —
  TIFF: —


In [16]:
print("First 64 bytes (no known magic):")
print(hexdump(spi, limit=64))
print("\nVerdict: Samsung proprietary bitmap \u2014 not decodable with standard tools.")

First 64 bytes (no known magic):
  0000  14 00 00 00 aa 01 00 00 00 14 00 00 00 00 00 07  ················
  0010  38 0f 4f 04 00 3e 00 e0 83 ec 06 00 aa 02 00 02  8·O··>··········
  0020  7b ea 00 00 00 00 43 02 e0 a0 2c 00 00 00 3f ff  {·····C···,···?·
  0030  ff ff ff ff ff ff ff ff ff ff ff ff ff ff ff ff  ················
  ... (453727 more bytes)

Verdict: Samsung proprietary bitmap — not decodable with standard tools.


---

## The `.page` File — Preview

The `.page` file contains all stroke geometry and attributes. At ~4.5 MB, it's by far the
largest file in the archive. Its detailed structure is covered in the next two notebooks.

In [17]:
page_key = [k for k in files if k.endswith(".page")][0]
page_data = files[page_key]
print(f"{page_key} \u2014 {len(page_data):,} bytes\n")

print("--- Header (first 128 bytes) ---")
print(hexdump(page_data, limit=128))

a87cbcbc-43ce-11ee-9848-0b8ce2c9daf6.page — 4,487,877 bytes

--- Header (first 128 bytes) ---
  0000  e3 00 00 00 80 00 00 00 04 00 00 00 00 04 71 04  ··············q·
  0010  00 00 00 00 00 00 38 07 00 00 9e 1e 00 00 00 00  ······8·········
  0020  00 00 00 00 00 00 24 00 61 00 38 00 37 00 63 00  ······$·a·8·7·c·
  0030  62 00 63 00 62 00 63 00 2d 00 34 00 33 00 63 00  b·c·b·c·-·4·3·c·
  0040  65 00 2d 00 31 00 31 00 65 00 65 00 2d 00 39 00  e·-·1·1·e·e·-·9·
  0050  38 00 34 00 38 00 2d 00 30 00 62 00 38 00 63 00  8·4·8·-·0·b·8·c·
  0060  65 00 32 00 63 00 39 00 64 00 61 00 66 00 36 00  e·2·c·9·d·a·f·6·
  0070  8a b4 2e ab 87 36 06 00 a0 0f 00 00 a0 0f 00 00  ··.··6··········
  ... (4487749 more bytes)


In [18]:
print("--- Footer (last 32 bytes) ---")
print(hexdump(page_data[-32:], offset=len(page_data) - 32))
print(f"\nFooter marker: {page_data[-26:].decode('ascii')!r}")

--- Footer (last 32 bytes) ---
  447aa5  08 88 c5 e8 6c a8 50 61 67 65 20 66 6f 72 20 53  ····l·Page for S
  447ab5  41 4d 53 55 4e 47 20 53 2d 50 65 6e 20 53 44 4b  AMSUNG S-Pen SDK

Footer marker: 'Page for SAMSUNG S-Pen SDK'


---

## Summary

| File | Size | Purpose |
|------|------|---------|
| `end_tag.bin` | 144 B | Document footer — created/modified timestamps (i64 ms epoch) |
| `pageIdInfo.dat` | 140 B | Page UUID registry with 32-byte integrity hashes |
| `media/mediaInfo.dat` | 131 B | Media manifest — filename (UTF-16LE) + SHA-256 verification |
| `note.note` | ~1.7 KB | Note title, pen tool names, background color, layer UUIDs |
| `media/*.spi` | ~454 KB | Page thumbnail — Samsung proprietary format, not decodable |
| `<uuid>.page` | ~4.5 MB | Stroke geometry and attributes — see notebooks 02 and 03 |

Each file has a recognizable footer marker:
- `end_tag.bin` → `"Document for S-Pen SDK"`
- `<uuid>.page` → `"Page for SAMSUNG S-Pen SDK"`
- `mediaInfo.dat` → `"EOFX"`

**Next:** [02_strokes.ipynb](02_strokes.ipynb) dives into the `.page` file to decode stroke geometry.